# Tenacious-Bench v0.1 — SimPO LoRA Training (Unsloth)

**Path B**: SimPO preference optimization on Qwen3.5-0.8B via Unsloth.

- Unsloth uses ~50% less VRAM and runs ~1.5x faster than vanilla TRL
- 4-bit quantization fits comfortably on free Colab T4 (15.6 GB)
- Backbone: `unsloth/Qwen3.5-0.8B` (pinned per challenge spec)

**Target metrics**:
- Delta A (dual_control_decision): ≥ +15 points on dev
- Delta B (signal_grounding): ≥ +10 points on dev

In [ ]:
# Step 1 — Install Unsloth (handles transformers/trl/peft versions automatically)
!pip install -q unsloth
!pip install -q --upgrade --no-deps unsloth_zoo

# Restart runtime after install so numpy loads cleanly
import os
os.kill(os.getpid(), 9)

In [ ]:
# Step 2 — Verify GPU (run after runtime restarts)
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Step 3 — Upload the 3 required files (one dialog per file)
from google.colab import files
import os, json

print("1/3 — Navigate to: week11/training_data/preference_pairs.jsonl")
files.upload()

print("2/3 — Navigate to: week11/tenacious_bench_v0.1/dev/dev.jsonl")
files.upload()

print("3/3 — Navigate to: week11/scoring_evaluator.py")
files.upload()

PAIRS_FILE = 'preference_pairs.jsonl'

for f in ['preference_pairs.jsonl', 'dev.jsonl', 'scoring_evaluator.py']:
    print(f"  {'OK' if os.path.exists(f) else 'MISSING'} — {f}")

In [ ]:
# Step 4 — Load and inspect preference pairs
from datasets import Dataset
from collections import Counter

pairs = [json.loads(l) for l in open(PAIRS_FILE)]
print(f"Loaded {len(pairs)} preference pairs")

dims = Counter(p['dimension'] for p in pairs)
for d, n in sorted(dims.items()):
    print(f"  {d:40s} {n}")

dataset = Dataset.from_list([
    {'prompt': p['prompt'], 'chosen': p['chosen'], 'rejected': p['rejected']}
    for p in pairs
])
print(f"\nDataset: {dataset}")

In [ ]:
# Step 5 — Load model with Unsloth (4-bit, gradient checkpointing built in)
from unsloth import FastLanguageModel

MAX_SEQ_LENGTH = 1024
MODEL_ID = "unsloth/Qwen3.5-0.8B"  # pinned backbone per challenge spec

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,       # fits T4 with room to spare
    dtype=None,              # auto-detect (fp16 on T4)
)

print(f"Model loaded: {sum(p.numel() for p in model.parameters())/1e6:.0f}M parameters")

In [ ]:
# Step 6 — Attach LoRA adapter via Unsloth
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",  # Unsloth's memory-efficient checkpointing
    random_state=42,
)
model.print_trainable_parameters()

In [ ]:
# Step 7 — Configure SimPO trainer
from trl import CPOConfig, CPOTrainer

training_args = CPOConfig(
    output_dir='./checkpoints',
    loss_type='simpo',
    simpo_gamma=0.5,
    beta=2.0,
    learning_rate=5e-5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,   # effective batch = 16
    max_steps=500,
    warmup_steps=50,
    lr_scheduler_type='cosine',
    fp16=True,
    logging_steps=20,
    save_steps=250,
    max_length=MAX_SEQ_LENGTH,
    max_prompt_length=768,
    remove_unused_columns=False,
    report_to='none',
    optim='adamw_8bit',              # Unsloth's 8-bit optimizer
)

trainer = CPOTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    tokenizer=tokenizer,
)

print("Trainer configured. Starting training...")

In [ ]:
# Step 8 — Train (~15 min on T4)
train_result = trainer.train()
print(train_result)

In [ ]:
# Step 9 — Save LoRA adapter
ADAPTER_DIR = './tenacious-simpo-lora'
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"Adapter saved to {ADAPTER_DIR}")

## Ablations — Delta A and Delta B

In [ ]:
# Step 10 — Baseline: score dev tasks using original candidate outputs
import sys
sys.path.insert(0, '.')
import scoring_evaluator as se

dev_tasks = [json.loads(l) for l in open('dev.jsonl')]
print(f"Dev tasks: {len(dev_tasks)}")

dev_dc = [t for t in dev_tasks if t.get('dimension') == 'dual_control_decision']
dev_sg = [t for t in dev_tasks if t.get('dimension') == 'signal_grounding']
print(f"  dual_control_decision: {len(dev_dc)}")
print(f"  signal_grounding:      {len(dev_sg)}")

def avg_score(tasks):
    scores = [se.score_task(t).get('total_score', 0) for t in tasks]
    return sum(scores)/len(scores) if scores else 0

baseline_dc = avg_score(dev_dc)
baseline_sg = avg_score(dev_sg)
print(f"\nBaseline avg — dual_control: {baseline_dc:.3f}  signal_grounding: {baseline_sg:.3f}")

In [ ]:
# Step 11 — Post-training: generate outputs and score
import re

FastLanguageModel.for_inference(model)  # Unsloth inference mode (2x faster)

def build_prompt(task):
    inp = task.get('input', {})
    ctx = inp.get('prospect_context', {})
    lines = [
        f"Company: {ctx.get('company_name', '')}",
        f"Funding: {ctx.get('funding_amount', '')} {ctx.get('funding_round', '')}",
        f"Employees: {ctx.get('employee_count', '')}",
        f"Engineering roles open: {ctx.get('engineering_roles_open', '')}",
        f"AI maturity score: {ctx.get('ai_maturity_score', '')}",
    ]
    reply = inp.get('prospect_reply')
    if reply:
        lines.append(f"Prospect reply: {reply}")
        lines.append(f"Reply intent: {inp.get('reply_intent', '')}")
        lines.append(f"Confidence: {inp.get('confidence', '')}")
    lines.append("\nDecide the correct action and generate the output.")
    return "\n".join(lines)

def generate_output(task):
    prompt = build_prompt(task)
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=768).to('cuda')
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=200,
                             do_sample=False, pad_token_id=tokenizer.eos_token_id)
    text = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    action_m = re.search(r'ACTION:\s*(\S+)', text)
    auto_m   = re.search(r'AUTONOMOUS:\s*(True|False)', text)
    email_m  = re.search(r'EMAIL:\n(.+?)(?=\n[A-Z_]+:|$)', text, re.DOTALL)
    esc_m    = re.search(r'ESCALATION_REASON:\s*(.+)', text)
    return {
        'action':           action_m.group(1) if action_m else 'unknown',
        'autonomous':       auto_m.group(1) == 'True' if auto_m else False,
        'email_body':       email_m.group(1).strip() if email_m else None,
        'subject_line':     None,
        'capacity_claim':   None,
        'escalation_reason': esc_m.group(1).strip() if esc_m else None,
    }

def score_with_model(tasks):
    scores = []
    for t in tasks:
        orig = t.get('candidate_output')
        t['candidate_output'] = generate_output(t)
        scores.append(se.score_task(t).get('total_score', 0))
        t['candidate_output'] = orig
    return sum(scores)/len(scores) if scores else 0

print("Evaluating model on dev (this takes a few minutes)...")
trained_dc = score_with_model(dev_dc)
trained_sg = score_with_model(dev_sg)

delta_a = (trained_dc - baseline_dc) * 100
delta_b = (trained_sg - baseline_sg) * 100

print(f"\n=== Ablation Results ===")
print(f"  Baseline dual_control:    {baseline_dc:.3f}")
print(f"  Trained  dual_control:    {trained_dc:.3f}")
print(f"  Delta A:                 {delta_a:+.1f} pts  {'PASS (>=15)' if delta_a >= 15 else 'below target (>=15)'}")
print()
print(f"  Baseline signal_grounding: {baseline_sg:.3f}")
print(f"  Trained  signal_grounding: {trained_sg:.3f}")
print(f"  Delta B:                  {delta_b:+.1f} pts  {'PASS (>=10)' if delta_b >= 10 else 'below target (>=10)'}")

In [ ]:
# Step 12 — Save ablation results and download
ablation_results = {
    'model': MODEL_ID,
    'lora_config': {'r': 16, 'lora_alpha': 32, 'lora_dropout': 0.05},
    'simpo_config': {'gamma': 0.5, 'beta': 2.0, 'lr': 5e-5, 'steps': 500},
    'training_pairs': len(pairs),
    'baseline': {'dual_control_avg': round(baseline_dc, 4), 'signal_grounding_avg': round(baseline_sg, 4)},
    'post_training': {'dual_control_avg': round(trained_dc, 4), 'signal_grounding_avg': round(trained_sg, 4)},
    'delta_a_points': round(delta_a, 2),
    'delta_b_points': round(delta_b, 2),
    'targets_met': {'delta_a_ge_15': delta_a >= 15, 'delta_b_ge_10': delta_b >= 10},
    'train_metrics': train_result.metrics if hasattr(train_result, 'metrics') else {},
}

with open('ablation_results.json', 'w') as f:
    json.dump(ablation_results, f, indent=2)

print(json.dumps(ablation_results, indent=2))

# Download results
files.download('ablation_results.json')
print("\nPlace ablation_results.json in: week11/ablations/")